## 1 读取负荷数据和光伏数据

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from gymnasium import spaces
import warnings
warnings.filterwarnings("ignore")

#设置随机种子
def set_random_seed(seed_value):
    """设置随机种子"""
    np.random.seed(seed_value)  # NumPy
    random.seed(seed_value)  # Python
    torch.manual_seed(seed_value)  # PyTorch CPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)  # PyTorch GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

#设置随机种子，确保结果可复现⭐⭐⭐
set_random_seed(42)
#设置设备
device='cuda' if torch.cuda.is_available() else 'cpu'

#读取数据
data_file = "Annual_load_PV_15min.csv"  # CSV 文件路径,此处为相对路径
df = pd.read_csv(data_file, parse_dates=["Time"])
df.set_index("Time", inplace=True)

pv_profile = df["PV_kW"].values.astype(np.float32)  # 光伏功率序列
load_profile = df["Load_kW"].values.astype(np.float32)  # 负载功率序列

# 爱沙尼亚峰谷电价(€/kWh)
price_profile = np.where((df.index.hour >= 17) & (df.index.hour <= 20),
                         0.30,  # 高峰
                         0.20).astype(np.float32)  # 谷时


## 2 环境定义

In [ ]:
# 定义储能环境类，继承自 gym.Env
class EnergyStorageEnv(gym.Env):
    def __init__(self, pv, load, price, e_cap=10.0, p_max=3.0, dt=0.25):
        super().__init__()  
        self.pv = pv        # 光伏发电功率序列
        self.load = load    # 负荷功率序列
        self.price = price  # 电价序列
        self.T = len(pv)    # 总时间步数（假设pv、load、price长度一致）
        self.dt = dt        # 每个时间步的时长（小时）
        self.e_cap = e_cap  # 储能容量（kWh）
        self.p_max = p_max  # 储能充放电最大功率（kW）
        self.soc_min = 0.1  # 储能最小SOC（状态的下限）
        self.soc_max = 0.9  # 储能最大SOC（状态的上限）

        # 定义观测空间（state），包含3个变量：SOC、负荷、光伏功率
        self.observation_space = spaces.Box(
            low=np.array([0.,0.,0.], dtype=np.float32),  # 各变量最小值
            high=np.array([1., np.max(load), np.max(pv)], dtype=np.float32),  # 各变量最大值
            dtype=np.float32
        )

        # 定义动作空间（action），0：放电，1：保持，2：充电
        self.action_space = spaces.Discrete(3)

    # 重置环境
    def reset(self):
        self.t = 0          # 当前时间步初始化为0
        self.soc = 0.5      # 初始电池SOC设为50%
        self.soc_history = []  # 用于记录每步SOC变化
        self.grid_history = [] # 用于记录每步网电进口功率

        # 返回初始观测值
        return np.array([self.soc, self.load[self.t], self.pv[self.t]], dtype=np.float32), {}

    # 环境一步推进函数
    def step(self, action):
        # 定义动作与功率映射：0->放电，1->保持，2->充电
        p_map = {0:-self.p_max, 1:0.0, 2:self.p_max}
        p_batt = p_map[action]  # 储能功率

        # 更新SOC，保证在最小/最大SOC之间
        self.soc = np.clip(self.soc + p_batt*self.dt/self.e_cap,
                           self.soc_min, self.soc_max)

        # 计算净负荷：负荷 - 光伏 - 储能功率
        net_load = self.load[self.t] - self.pv[self.t] - p_batt
        grid_import = max(net_load, 0)  # 如果净负荷为负，则不用从电网购电

        # 奖励函数：负的网电成本（希望最小化电网购电）
        reward = -grid_import*self.price[self.t]*self.dt

        # 记录SOC和网电历史
        self.soc_history.append(self.soc)
        self.grid_history.append(grid_import)

        # 时间步增加
        self.t += 1

        # 判断是否结束
        terminated = self.t >= self.T
        truncated = False  # Gym 0.26+ API 需要返回truncated

        # 生成下一个观测值，如果已结束则用0填充
        obs = np.array([self.soc,
                        self.load[self.t] if not terminated else 0,
                        self.pv[self.t] if not terminated else 0], dtype=np.float32)
        
        # 返回：观测值、奖励、是否结束、截断标记、额外信息
        return obs, reward, terminated, truncated, {}

# 创建环境实例，传入光伏、负荷、电价序列
env = EnergyStorageEnv(pv_profile, load_profile, price_profile)

## 可视化的相关函数（四种策略均适用）

In [ ]:
# 训练过程中累计奖励的实时可视化
def plot_reward_live(rewards, title="Live Reward"):
    plt.figure(figsize=(8,4))
    plt.plot(rewards, marker='o', color='blue')
    plt.title(title)
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.grid(True)
    plt.show()

# SOC、Reward、Grid Power的可视化
def plot_training_animation(rewards, soc_history, grid_history,
                                     title="Training Animation",
                                     save_path=None, fps=5):
    """
    绘制训练过程动画，包含三个子图：
    1. Reward（奖励）
    2. SOC（电池状态）
    3. Grid Power（网电功率）
    
    参数：
    rewards     : list或array，每步训练的奖励值
    soc_history : list或array，每步的电池SOC值
    grid_history: list或array，每步的网电进口功率
    title       : 图的总标题（可选）
    save_path   : 保存动画路径（可选）
    fps         : 保存动画帧率（可选）
    """
    
    # 创建三个垂直排列的子图，共享x轴
    fig, axes = plt.subplots(3,1, figsize=(10,8), sharex=True)

    # 在三个子图上初始化空曲线，用于动画更新
    line_reward, = axes[0].plot([], [], color='blue', label='Reward', marker='o')  # 奖励曲线
    line_soc, = axes[1].plot([], [], color='green', label='SOC')                   # SOC曲线
    line_grid, = axes[2].plot([], [], color='red', label='Grid Power (kW)')       # 网电功率曲线

    # 配置第一个子图：奖励
    axes[0].set_ylabel("Reward")   # y轴标签
    axes[0].grid(True)             # 显示网格
    axes[0].set_title("Training Reward")  # 子图标题

    # 配置第二个子图：SOC
    axes[1].set_ylabel("SOC")      # y轴标签
    axes[1].grid(True)             # 显示网格
    axes[1].set_title("Battery SOC")  # 子图标题

    # 配置第三个子图：网电功率
    axes[2].set_ylabel("Grid Power (kW)")  # y轴标签
    axes[2].set_xlabel("Episode / Time Step")  # x轴标签
    axes[2].grid(True)                     # 显示网格
    axes[2].set_title("Grid Power")        # 子图标题

    # 设置整个图的总标题
    fig.suptitle(title, fontsize=16, fontweight='bold')

    # 自动调整子图布局，避免标题重叠
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # 动画初始化函数，每次开始时将所有曲线清空
    def init():
        line_reward.set_data([], [])
        line_soc.set_data([], [])
        line_grid.set_data([], [])
        return line_reward, line_soc, line_grid

    # 动画更新函数，每帧绘制到第i步
    def update(i):
        x = range(i+1)  # x轴为时间步序列
        line_reward.set_data(x, rewards[:i+1])        # 更新奖励曲线数据
        line_soc.set_data(x, soc_history[:i+1])       # 更新SOC曲线数据
        line_grid.set_data(x, grid_history[:i+1])     # 更新网电功率曲线数据
        return line_reward, line_soc, line_grid

    # 创建动画对象
    ani = FuncAnimation(fig, update, frames=len(rewards),
                        init_func=init, interval=200, blit=True)

    # 在Jupyter Notebook中显示动画
    display(HTML(ani.to_jshtml()))
    plt.close(fig)  # 关闭静态图，避免重复显示

    # 如果指定保存路径，则将动画保存为视频
    if save_path:
        ani.save(save_path, writer="ffmpeg", fps=fps)

## 3 Q-Learning 算法

In [ ]:

# -------------------------------
# 1. 离散化状态空间
# 将连续的SOC、负荷、光伏功率划分为离散区间
soc_bins = np.linspace(env.soc_min, env.soc_max, 8)         # 将SOC划分为8个等间隔区间
load_bins = np.linspace(0, max(load_profile), 12)           # 将负荷划分为12个等间隔区间
pv_bins = np.linspace(0, max(pv_profile), 12)               # 将光伏功率划分为12个等间隔区间

# -------------------------------
# 2. 状态离散化函数
def discretize_state(s):
    """
    输入连续状态 s = [SOC, load, pv]
    输出对应离散状态索引
    np.digitize返回每个值所属的区间索引（1-based），这里减1转为0-based索引
    """
    return (np.digitize(s[0], soc_bins)-1,
            np.digitize(s[1], load_bins)-1,
            np.digitize(s[2], pv_bins)-1)

# -------------------------------
# 3. 初始化 Q-table
# Q-table 维度为 (SOC_bins, load_bins, pv_bins, 动作数)
q_table = np.zeros((8,12,12,3))

# -------------------------------
# 4. 设置 Q-learning 参数
lr = 0.1      # 学习率
gamma = 0.99  # 折扣因子
eps = 0.1     # epsilon-greedy 探索率

# 用于记录训练过程数据
q_rewards, q_soc_history, q_grid_history = [], [], []

# -------------------------------
# 5. 设置训练总轮数
total_episodes = 50

# -------------------------------
# 6. Q-learning 主训练循环
for ep in range(total_episodes):
    # 重置环境，获得初始连续状态
    s, _ = env.reset()
    # 离散化初始状态
    idx = discretize_state(s)
    done = False
    total_reward = 0
    # 初始化每轮的SOC和网电记录
    env.soc_history, env.grid_history = [], []

    # -------------------------------
    # 每个 episode 内循环
    while not done:
        # epsilon-greedy 策略选择动作
        if random.random() < eps:
            a = env.action_space.sample()  # 随机选择动作（探索）
        else:
            a = np.argmax(q_table[idx])    # 选择当前状态下Q值最大的动作（利用）

        # 执行动作，获得下一状态和奖励
        ns, r, terminated, truncated, _ = env.step(a)
        # 判断是否结束
        done = terminated or truncated
        # 离散化下一状态
        nidx = discretize_state(ns)
        # Q-learning 更新公式
        q_table[idx][a] += lr * (r + gamma*np.max(q_table[nidx]) - q_table[idx][a])
        # 状态转移
        idx = nidx
        # 累计奖励
        total_reward += r

    # -------------------------------
    # 每个 episode 结束后记录
    q_rewards.append(total_reward)          # 记录该 episode 总奖励
    q_soc_history = env.soc_history.copy()  # 记录该 episode SOC 历史
    q_grid_history = env.grid_history.copy()# 记录该 episode 网电功率历史

    # -------------------------------
    # 动态绘制每轮 reward（可在 Jupyter 中实时观察训练进度）
    clear_output(wait=True)                  # 清空上一次输出
    plot_reward_live(q_rewards, title="Q-Learning Live Reward")  # 实时绘图

# -------------------------------
# 7. 训练完成后绘制完整动画
plot_training_animation(q_rewards, q_soc_history, q_grid_history, title="Q-Learning Animation")

## 4 DQN算法

In [ ]:
# -------------------------------
# 1. 定义 DQN 网络
class DQNNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 一个简单的全连接网络，输入3维状态，隐藏层64，输出3个动作的Q值
        self.fc = nn.Sequential(
            nn.Linear(3,64),  # 输入3维（SOC, load, PV）
            nn.ReLU(),         # 激活函数
            nn.Linear(64,3)    # 输出3维动作Q值
        )

    # 前向传播
    def forward(self,x):
        return self.fc(x.float())  # 确保输入为浮点型

# -------------------------------
# 2. 创建 DQN 网络实例和目标网络
dqn_net = DQNNet()              # 在线网络
target_net = DQNNet()           # 目标网络
target_net.load_state_dict(dqn_net.state_dict())  # 初始化目标网络参数与在线网络一致

# -------------------------------
# 3. 优化器设置
optimizer = optim.Adam(dqn_net.parameters(), lr=1e-3)  # Adam优化器，学习率0.001

# -------------------------------
# 4. 经验回放缓冲区
memory = deque(maxlen=5000)  # 最大长度5000的双端队列
batch_size = 32              # 每次训练采样32条经验

# -------------------------------
# 5. 训练过程数据记录
dqn_rewards, dqn_soc_history, dqn_grid_history = [], [], []

# -------------------------------
# 6. DQN训练循环（每个episode）
for ep in range(total_episodes):
    s, _ = env.reset()  # 重置环境，获取初始状态
    done = False
    total_reward = 0
    env.soc_history, env.grid_history = [], []  # 每轮记录SOC和网电功率历史

    # -------------------------------
    # 每个episode的时间步循环
    while not done:
        # epsilon-greedy策略选择动作
        if random.random() < eps:
            a = env.action_space.sample()  # 随机选择动作（探索）
        else:
            # 使用在线网络选择Q值最大的动作（利用）
            a = torch.argmax(dqn_net(torch.tensor(s, dtype=torch.float32))).item()

        # 执行动作，获得下一个状态、奖励、是否结束
        ns, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated

        # 将下一状态转换为tensor
        ns_tensor = torch.tensor(ns, dtype=torch.float32)
        # 将经验加入回放缓冲区
        memory.append((torch.tensor(s, dtype=torch.float32), a, r, ns_tensor, done))

        # 状态更新
        s = ns
        # 累计奖励
        total_reward += r

        # -------------------------------
        # 当回放缓冲区经验数 >= batch_size 时，进行一次网络更新
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)       # 随机采样batch_size条经验
            states, actions, rewards_batch, next_states, dones = zip(*batch)  # 解包经验
            states = torch.stack(states).float()           # 合并为tensor
            next_states = torch.stack(next_states).float() # 合并下一状态
            actions = torch.tensor(actions)               # 动作tensor
            rewards_batch = torch.tensor(rewards_batch)   # 奖励tensor
            dones = torch.tensor(dones, dtype=torch.float32)  # done标记

            # -------------------------------
            # 在线网络预测当前Q值
            q_values = dqn_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

            # -------------------------------
            # 目标网络预测下一状态最大Q值
            with torch.no_grad():
                q_next = target_net(next_states).max(1)[0]

            # -------------------------------
            # 计算目标Q值
            target = rewards_batch + gamma * q_next * (1 - dones)

            # -------------------------------
            # 计算损失并梯度更新
            loss = nn.MSELoss()(q_values, target)  # 均方误差
            optimizer.zero_grad()                   # 梯度清零
            loss.backward()                         # 反向传播
            optimizer.step()                        # 更新参数

    # -------------------------------
    # 记录每轮episode的数据
    dqn_rewards.append(total_reward)
    dqn_soc_history = env.soc_history.copy()
    dqn_grid_history = env.grid_history.copy()

    # -------------------------------
    # 动态绘制奖励曲线
    clear_output(wait=True)  # 清除上一次输出
    plot_reward_live(dqn_rewards, title="DQN Live Reward")  # 实时绘图

# -------------------------------
# 训练完成后绘制完整动画
plot_training_animation(dqn_rewards, dqn_soc_history, dqn_grid_history, title="DQN Animation")

## 5 Policy Gradient

In [ ]:
# -------------------------------
# 1. 定义策略网络（Policy Network）
class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 全连接网络：输入3维状态 -> 隐藏层64 -> 输出3维动作概率 -> Softmax归一化
        self.fc = nn.Sequential(
            nn.Linear(3,64),     # 输入3维状态（SOC, load, PV）
            nn.ReLU(),           # 激活函数
            nn.Linear(64,3),     # 输出动作得分（logits）
            nn.Softmax(dim=-1)   # 将得分转为动作概率分布
        )
    
    # 前向传播
    def forward(self,x):
        return self.fc(x.float())  # 确保输入为浮点型

# -------------------------------
# 2. 创建策略网络实例
policy_net = PolicyNet()
# 优化器
optimizer_pg = optim.Adam(policy_net.parameters(), lr=1e-3)

# -------------------------------
# 3. 训练过程记录
pg_rewards, pg_soc_history, pg_grid_history = [], [], []

# -------------------------------
# 4. 策略梯度训练循环
for ep in range(total_episodes):
    s, _ = env.reset()  # 重置环境
    done = False
    log_probs, rewards_list = [], []  # 用于存储每步动作的log_prob和奖励
    total_reward = 0
    env.soc_history, env.grid_history = [], []  # 每轮记录SOC和网电功率

    # -------------------------------
    # 每个episode的时间步循环
    while not done:
        # 计算当前状态下动作概率
        probs = policy_net(torch.tensor(s, dtype=torch.float32))
        # 根据概率构建类别分布
        dist = torch.distributions.Categorical(probs)
        # 从分布中采样动作
        a = dist.sample()
        # 执行动作
        ns, r, terminated, truncated, _ = env.step(a.item())
        done = terminated or truncated
        # 记录当前动作的log_prob
        log_probs.append(dist.log_prob(a))
        # 记录奖励
        rewards_list.append(r)
        # 状态更新
        s = ns
        # 累计奖励
        total_reward += r

    # -------------------------------
    # 计算回报（Returns）
    G = 0
    returns = []
    # 从最后一步开始反向累积折扣回报
    for r in reversed(rewards_list):
        G = r + gamma*G
        returns.insert(0,G)  # 插入到列表开头
    returns = torch.tensor(returns)
    # 对回报进行归一化，提高训练稳定性
    returns = (returns - returns.mean()) / (returns.std() + 1e-9)

    # -------------------------------
    # 计算策略梯度损失
    loss = 0
    for lp, R in zip(log_probs, returns):
        loss -= lp * R  # 策略梯度目标：maximize log_prob*return = minimize -log_prob*return

    # -------------------------------
    # 梯度更新
    optimizer_pg.zero_grad()  # 清空梯度
    loss.backward()           # 反向传播计算梯度
    optimizer_pg.step()       # 参数更新

    # -------------------------------
    # 每轮episode数据记录
    pg_rewards.append(total_reward)
    pg_soc_history = env.soc_history.copy()
    pg_grid_history = env.grid_history.copy()

    # -------------------------------
    # 动态绘制奖励曲线
    clear_output(wait=True)  # 清除上一次输出
    plot_reward_live(pg_rewards, title="Policy Gradient Live Reward")  # 实时绘图

# -------------------------------
# 5. 训练完成后绘制完整动画
plot_training_animation(pg_rewards, pg_soc_history, pg_grid_history, title="Policy Gradient Animation")

## 6 Actor-Critic(AC) 演员-评论家算法

In [ ]:
# -------------------------------
# 1. 定义 Actor-Critic 网络
class ACNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 公共隐藏层：输入3维状态 -> 64维隐藏
        self.fc = nn.Linear(3,64)
        # Actor头：输出每个动作的概率（3个动作）
        self.actor = nn.Linear(64,3)
        # Critic头：输出状态价值（标量）
        self.critic = nn.Linear(64,1)

    # 前向传播
    def forward(self,x):
        x = torch.relu(self.fc(x.float()))         # 先经过隐藏层+ReLU
        return torch.softmax(self.actor(x),-1), self.critic(x)  # Actor输出动作概率，Critic输出状态值

# -------------------------------
# 2. 创建 Actor-Critic 网络实例
ac_net = ACNet()
# 优化器
optimizer_ac = optim.Adam(ac_net.parameters(), lr=1e-3)

# -------------------------------
# 3. 训练过程记录
ac_rewards, ac_soc_history, ac_grid_history = [], [], []

# -------------------------------
# 4. Actor-Critic训练循环
for ep in range(total_episodes):
    s, _ = env.reset()  # 重置环境
    done = False
    total_reward = 0
    env.soc_history, env.grid_history = [], []  # 每轮记录SOC和网电功率

    # -------------------------------
    # 每个episode的时间步循环
    while not done:
        # 输入当前状态，得到动作概率和状态值
        probs, value = ac_net(torch.tensor(s, dtype=torch.float32))
        # 根据概率构建类别分布
        dist = torch.distributions.Categorical(probs)
        # 从分布中采样动作
        a = dist.sample()
        # 执行动作
        ns, r, terminated, truncated, _ = env.step(a.item())
        done = terminated or truncated

        # -------------------------------
        # 计算下一状态的状态值（Critic）
        _, next_value = ac_net(torch.tensor(ns, dtype=torch.float32))

        # -------------------------------
        # TD目标：r + gamma * V(s') * (1 - done)
        td_target = r + gamma * next_value * (1 - done)
        # TD误差（优势函数估计）
        td_error = td_target - value

        # -------------------------------
        # Actor-Critic损失
        # Actor部分：最大化 log_prob * td_error
        # Critic部分：最小化 td_error^2
        loss = -dist.log_prob(a) * td_error.detach() + td_error.pow(2)

        # -------------------------------
        # 梯度更新
        optimizer_ac.zero_grad()  # 清空梯度
        loss.backward()           # 反向传播
        optimizer_ac.step()       # 参数更新

        # 状态更新
        s = ns
        # 累计奖励
        total_reward += r

    # -------------------------------
    # 每轮episode数据记录
    ac_rewards.append(total_reward)
    ac_soc_history = env.soc_history.copy()
    ac_grid_history = env.grid_history.copy()

    # -------------------------------
    # 动态绘制奖励曲线
    clear_output(wait=True)  # 清除上一次输出
    plot_reward_live(ac_rewards, title="Actor-Critic Live Reward")  # 实时绘图

# -------------------------------
# 5. 训练完成后绘制完整动画
plot_training_animation(ac_rewards, ac_soc_history, ac_grid_history, title="Actor-Critic Animation")